# MADDPG Baseline — Results

**Task:** `simple_tag_v3` — 3 chasers vs 1 evader  
**Algorithm:** MADDPG (centralized training, decentralized execution)  

Figures are saved to the run directory alongside `metrics.csv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from pathlib import Path

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.frameon": False,
})

COLORS = {"adversary": "#2196F3", "agent": "#F44336"}
SMOOTH_WINDOW = 10  # rolling mean window for noisy train metrics
fmt_ksteps = mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}k")

In [ ]:
# Finds the most recent run that has a complete metrics.csv (with eval columns).
# Override by setting RUN_DIR explicitly, e.g.:
# RUN_DIR = Path("runs/20260319_210851")

EVAL_COLS = ["eval_ep_return_adversary", "eval_ep_return_agent"]

def try_read_csv(path: Path) -> pd.DataFrame | None:
    """Read a CSV, returning None if the file is malformed."""
    try:
        return pd.read_csv(path)
    except Exception:
        return None

def find_latest_run(runs_root: Path) -> tuple[Path, pd.DataFrame]:
    for run_dir in sorted(runs_root.iterdir(), reverse=True):
        csv = run_dir / "metrics.csv"
        if not csv.exists():
            continue
        df = try_read_csv(csv)
        if df is None:
            print(f"Skipping malformed CSV: {csv}")
            continue
        if not all(c in df.columns for c in EVAL_COLS):
            continue
        if df.dropna(subset=EVAL_COLS).empty:      # skip runs with no eval data
            print(f"Skipping run with no eval data: {run_dir.name}")
            continue
        return run_dir, df
    raise FileNotFoundError(f"No complete run with eval columns found in {runs_root}")

RUN_DIR, df = find_latest_run(Path("runs"))
print(f"Run: {RUN_DIR}  |  {len(df)} iterations")

eval_df = df.dropna(subset=EVAL_COLS).copy()
print(f"Eval checkpoints: {len(eval_df)}")

def smooth(s):
    return s.rolling(SMOOTH_WINDOW, min_periods=1, center=True).mean()

df.tail(3)

## Figure 1 — Eval Learning Curve
Episode return under the deterministic policy, averaged over 10 episodes every 10 training iterations.  
This is the primary comparable metric (matches the MADDPG paper reporting convention).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for group, label in [("adversary", "Adversary (chasers)"), ("agent", "Agent (evader)")]:
    ax.plot(
        eval_df["total_frames"],
        eval_df[f"eval_ep_return_{group}"],
        marker="o", markersize=4,
        color=COLORS[group], label=label,
    )

ax.set_xlabel("Environment Steps")
ax.set_ylabel("Episode Return")
ax.set_title("Eval Episode Return (Deterministic Policy)")
ax.xaxis.set_major_formatter(fmt_ksteps)
ax.legend()

plt.tight_layout()
plt.savefig(RUN_DIR / "fig_eval_returns.png", bbox_inches="tight")
plt.show()

## Figure 2 — Train Episode Return
Per-iteration return computed from the collector batch (includes exploration noise).  
Faint line = raw, solid line = rolling mean.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for group, label in [("adversary", "Adversary (chasers)"), ("agent", "Agent (evader)")]:
    col = f"return_{group}" if f"return_{group}" in df.columns else f"ep_return_{group}"
    ax.plot(df["total_frames"], df[col], alpha=0.15, color=COLORS[group])
    ax.plot(df["total_frames"], smooth(df[col]), color=COLORS[group], label=label)

ax.set_xlabel("Environment Steps")
ax.set_ylabel("Episode Return")
ax.set_title("Train Episode Return (with smoothing)")
ax.xaxis.set_major_formatter(fmt_ksteps)
ax.legend()

plt.tight_layout()
plt.savefig(RUN_DIR / "fig_train_returns.png", bbox_inches="tight")
plt.show()

## Figure 3 — Training Diagnostics
Heuristic metrics computed from raw collector observations each iteration.  
Useful for understanding *how* the agents are learning to coordinate.

In [ ]:
diagnostics = [
    ("capture_rate",    "Capture Rate",         "Fraction of Episodes"),
    ("time_to_capture", "Time to First Capture", "Normalized Step (0=start, 1=end)"),
    ("collision_rate",  "Predator Collisions",   "Collisions / Step"),
    ("coverage_eff",    "Coverage Efficiency",   "Mean Pairwise Distance"),
]

# Only plot diagnostics that are present in this run's CSV
available = [(col, title, ylabel) for col, title, ylabel in diagnostics if col in df.columns]
if not available:
    print("No diagnostic metrics in this run's CSV — skipping Figure 3.")
else:
    fig, axes = plt.subplots(1, len(available), figsize=(5 * len(available), 4))
    if len(available) == 1:
        axes = [axes]

    for ax, (col, title, ylabel) in zip(axes, available):
        series = df[col].dropna()
        frames = df.loc[series.index, "total_frames"]
        ax.plot(frames, series, alpha=0.15, color="#555")
        ax.plot(frames, smooth(series), color="#222")
        ax.set_title(title)
        ax.set_ylabel(ylabel)
        ax.set_xlabel("Environment Steps")
        ax.xaxis.set_major_formatter(fmt_ksteps)

    plt.suptitle("Training Diagnostics", fontsize=13)
    plt.tight_layout()
    plt.savefig(RUN_DIR / "fig_diagnostics.png", bbox_inches="tight")
    plt.show()

## Multi-Seed Learning Curves (IQM ± 95% CI)

Loads all completed seed runs from `runs/`, computes the **Interquartile Mean** (mean of middle 50% of seeds) and a **bootstrapped 95% confidence interval** at each eval checkpoint.  
This matches the reporting convention from *Deep RL at the Edge of the Statistical Precipice* (Agarwal et al. 2021) used in MADDPG/BenchMARL comparisons.

In [ ]:
def load_all_seeds(runs_root: Path, eval_cols=EVAL_COLS) -> pd.DataFrame:
    """Load all complete, parseable run CSVs that contain eval columns."""
    frames = []
    for run_dir in sorted(runs_root.iterdir()):
        csv = run_dir / "metrics.csv"
        if not csv.exists():
            continue
        df = try_read_csv(csv)
        if df is None:
            print(f"Skipping malformed CSV: {csv}")
            continue
        if not all(c in df.columns for c in eval_cols):
            continue
        if df.dropna(subset=eval_cols).empty:
            continue
        df["run"] = run_dir.name
        frames.append(df)
    if not frames:
        raise FileNotFoundError(f"No complete runs found in {runs_root}")
    combined = pd.concat(frames, ignore_index=True)
    n_seeds = combined["run"].nunique()
    print(f"Loaded {n_seeds} seed(s): {sorted(combined['run'].unique())}")
    return combined

all_df   = load_all_seeds(Path("runs"))
all_eval = all_df.dropna(subset=EVAL_COLS).copy()

In [ ]:
def iqm(scores: np.ndarray) -> float:
    """Mean of the middle 50% of scores (interquartile mean)."""
    s = np.sort(scores)
    lo, hi = int(np.floor(len(s) * 0.25)), int(np.ceil(len(s) * 0.75))
    return float(np.mean(s[lo:hi]))

def bootstrap_iqm_ci(scores: np.ndarray, n_boot: int = 2000, alpha: float = 0.05):
    """Stratified bootstrap 95% CI around IQM."""
    rng = np.random.default_rng(0)
    boot = [iqm(rng.choice(scores, size=len(scores), replace=True)) for _ in range(n_boot)]
    return np.percentile(boot, [100 * alpha / 2, 100 * (1 - alpha / 2)])

In [ ]:
# Compute IQM + CI at each eval checkpoint across all seeds
checkpoints = sorted(all_eval["total_frames"].unique())
results = {}
for group in ["adversary", "agent"]:
    col = f"eval_ep_return_{group}"
    iqms, lo_cis, hi_cis = [], [], []
    for frame in checkpoints:
        vals = all_eval[all_eval["total_frames"] == frame][col].dropna().values
        if len(vals) == 0:
            iqms.append(np.nan); lo_cis.append(np.nan); hi_cis.append(np.nan)
        else:
            iqms.append(iqm(vals))
            lo, hi = bootstrap_iqm_ci(vals)
            lo_cis.append(lo); hi_cis.append(hi)
    results[group] = {
        "iqm": np.array(iqms),
        "lo":  np.array(lo_cis),
        "hi":  np.array(hi_cis),
    }

n_seeds = all_eval["run"].nunique()
frames  = np.array(checkpoints)

fig, ax = plt.subplots(figsize=(7, 4))
for group, label in [("adversary", "Adversary (chasers)"), ("agent", "Agent (evader)")]:
    r = results[group]
    ax.plot(frames, r["iqm"], color=COLORS[group], label=label,
            marker="o", markersize=4)
    ax.fill_between(frames, r["lo"], r["hi"], color=COLORS[group], alpha=0.2)

ax.set_xlabel("Environment Steps")
ax.set_ylabel("Episode Return (IQM)")
ax.set_title(f"Eval Episode Return — IQM ± 95% CI  ({n_seeds} seed{'s' if n_seeds > 1 else ''})")
ax.xaxis.set_major_formatter(fmt_ksteps)
ax.legend()

plt.tight_layout()
plt.savefig(Path("runs") / "fig_iqm_learning_curve.png", bbox_inches="tight")
plt.show()